In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day1_Telematics") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

In [30]:
from pyspark.sql.functions import when,avg,sum,max

In [2]:
df_tele = spark.read.csv("/opt/spark-data/telematics.csv", header=True, inferSchema=True)
df_vehicle = spark.read.csv("/opt/spark-data/vehicles.csv", header=True, inferSchema=True)

In [3]:
df_tele.show(5)

+-------------------+----------+-----+-----------+----------+--------+--------+
|          timestamp|vehicle_id|speed|engine_temp|fuel_level|     lat|     lon|
+-------------------+----------+-----+-----------+----------+--------+--------+
|2024-03-01 00:00:00|        V1|   41|         84|        50|13.72701|77.92426|
|2024-03-01 00:00:00|        V2|   17|        105|        95|13.86915|77.90112|
|2024-03-01 00:00:00|        V3|   79|         94|        79|13.06656|78.44125|
|2024-03-01 00:00:00|        V4|  100|        117|        44|13.57825|78.28303|
|2024-03-01 00:00:00|        V5|   80|         70|        83|13.74648|77.68761|
+-------------------+----------+-----+-----------+----------+--------+--------+
only showing top 5 rows



In [4]:
df_vehicle.show()

+----------+-----+-----+----+
|vehicle_id|model|plant|year|
+----------+-----+-----+----+
|        V1|Sedan|    A|2022|
|        V2|  SUV|    B|2023|
|        V3|Truck|    A|2021|
|        V4|Sedan|    C|2020|
|        V5|  SUV|    B|2022|
+----------+-----+-----+----+



## 🎯 Business Scenario

You are working for:

Fleet / IoT / Logistics company (Uber / Bosch / Caterpillar type)

They want a daily monitoring dataset to answer:

👉 Which vehicles are risky?
👉 Which plants/models are underperforming?
👉 Where should operations teams take action?

In [5]:
df = df_tele.join(df_vehicle,on='vehicle_id',how='inner')

In [14]:
df.show()

+----------+-------------------+-----+-----------+----------+--------+--------+-----+-----+----+
|vehicle_id|          timestamp|speed|engine_temp|fuel_level|     lat|     lon|model|plant|year|
+----------+-------------------+-----+-----------+----------+--------+--------+-----+-----+----+
|        V1|2024-03-01 00:00:00|   41|         84|        50|13.72701|77.92426|Sedan|    A|2022|
|        V2|2024-03-01 00:00:00|   17|        105|        95|13.86915|77.90112|  SUV|    B|2023|
|        V3|2024-03-01 00:00:00|   79|         94|        79|13.06656|78.44125|Truck|    A|2021|
|        V4|2024-03-01 00:00:00|  100|        117|        44|13.57825|78.28303|Sedan|    C|2020|
|        V5|2024-03-01 00:00:00|   80|         70|        83|13.74648|77.68761|  SUV|    B|2022|
|        V1|2024-03-01 00:05:00|   93|         98|        80|13.80148|78.04476|Sedan|    A|2022|
|        V2|2024-03-01 00:05:00|   90|        107|        65|13.04997|77.97242|  SUV|    B|2023|
|        V3|2024-03-01 00:05:0

### Identify Risky Vehicles and Score the level of Risk (Vehicles with Engine Temp above 100 or Fuel Level below 20)

In [22]:
df = df.withColumn("is_overspeed",when(df.speed > 90,1).otherwise(0)) \
    .withColumn("is_low_fuel",when(df.fuel_level<20,1).otherwise(0)) \
    .withColumn("is_overheat",when(df.engine_temp>110,1).otherwise(0))
        

In [42]:
df_risky_vehicles = df.groupBy("vehicle_id","model","plant").agg(avg("speed").alias("avg_Speed")
                                                                 ,avg("engine_temp").alias("avg_engine_temp")
                                                                 ,avg("fuel_level").alias("avg_fuel_level")
                                                                 ,sum("is_overspeed").alias("speeding_vehicles")
                                                                 ,sum("is_low_fuel").alias("fuel_low_events")
                                                                 ,sum("is_overheat").alias("overheated_vehicles")
                                                                )

In [43]:
df_risky_vehicles = df_risky_vehicles.withColumn("riskscore",(df_risky_vehicles.speeding_vehicles * 3) 
                                                             + (df_risky_vehicles.fuel_low_events * 2)
                                                             + (df_risky_vehicles.overheated_vehicles * 3)
                                                )
max_val = df_risky_vehicles.agg(max(df_risky_vehicles.riskscore)).collect()[0][0]

df_risky_vehicles = df_risky_vehicles.withColumn("riskscore",(df_risky_vehicles.riskscore/max_val)*10)

In [45]:
df_risky = df_risky_vehicles.filter(df_risky_vehicles.riskscore > 5)

In [46]:
df_risky.show(10)

+----------+-----+-----+------------------+-----------------+------------------+-----------------+---------------+-------------------+-----------------+
|vehicle_id|model|plant|         avg_Speed|  avg_engine_temp|    avg_fuel_level|speeding_vehicles|fuel_low_events|overheated_vehicles|        riskscore|
+----------+-----+-----+------------------+-----------------+------------------+-----------------+---------------+-------------------+-----------------+
|        V2|  SUV|    B|60.079166666666666|94.98159722222222|55.083333333333336|             2142|            951|               1697|9.863285556780596|
|        V1|Sedan|    A| 59.90127314814815|95.14502314814816|54.793055555555554|             2137|            972|               1750|             10.0|
|        V4|Sedan|    C| 59.89189814814815|94.86585648148149| 55.29907407407408|             2118|            921|               1701|9.775082690187432|
|        V5|  SUV|    B| 59.64039351851852|95.08449074074075| 54.89247685185185|  

### Identify Underperforming Models and Plants

In [59]:
df_poormodels = df.groupBy("plant","model").agg(avg("engine_temp").alias("avg_engine_temp")
                                                    ,sum("is_overheat").alias("overheated_vehicles"))
max_avg_engine_temp = df_poormodels.agg(max(df_poormodels.avg_engine_temp)).collect()[0][0]
max_overheat = df_poormodels.agg(max(df_poormodels.overheated_vehicles)).collect()[0][0]

In [58]:
max_avg_engine_temp

95.14502314814816

In [57]:
max_overheat

3413

In [60]:
df_poormodels = df_poormodels.withColumn("riskscore",100 - (((df_poormodels.avg_engine_temp/max_avg_engine_temp)*100*0.5) + ((df_poormodels.overheated_vehicles/max_overheat)*100*0.5)))

In [61]:
#df_poormodels = df_poormodels.filter(df_poormodels.riskscore > 5)
df_poormodels.show()

+-----+-----+-----------------+-------------------+-------------------+
|plant|model|  avg_engine_temp|overheated_vehicles|          riskscore|
+-----+-----+-----------------+-------------------+-------------------+
|    A|Truck|94.90358796296296|               1708|  25.10490266187834|
|    A|Sedan|95.14502314814816|               1750| 24.362730735423384|
|    C|Sedan|94.86585648148149|               1701| 25.227280144161114|
|    B|  SUV|95.03304398148148|               3413|0.05884657071989352|
+-----+-----+-----------------+-------------------+-------------------+

